In [1]:
import pandas as pd
import os
import ast
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
folder = "Data"
df = pd.concat([pd.read_csv(os.path.join(folder, f)) for f in os.listdir(folder) if f.endswith(".csv")], ignore_index=True) # Concat toutes les data dans un seul df
pd.set_option("display.max_colwidth", None)

In [60]:
df = df.drop(columns = ['photo_count', 'from_accepted_tag', 'achievement_count', 'location_state', 'kudos_count', 'comment_count', 'map', 'trainer', 'commute', 'flagged', 'start_latlng', 'end_latlng',
                        'upload_id', 'upload_id_str', 'external_id', 'from_accepted_tag', 'pr_count', 'total_photo_count', 'has_kudoed', 'location_country', 'location_city', 'start_date_local',
                        'display_hide_heartrate_option', 'elapsed_time', 'utc_offset', 'device_watts', 'workout_type', 'type', 'resource_state', 'timezone', 'elev_low', 'elev_high', 'manual', 'kilojoules',
                        'perceived_exertion', 'prefer_perceived_exertion', 'segment_efforts', 'description', 'stats_visibility', 'hide_from_home', 'photos', 'similar_activities', 'device_name', 'message',
                        'gear', 'gear_id', 'embed_token', 'errors', 'splits_standard', 'has_heartrate', 'available_zones', 'private', 'visibility', 'heartrate_opt_out', 'calories', 'average_temp', 'name', 
                        'athlete_count', 'average_cadence', 'best_efforts'

       ])

In [61]:
# === Data processing pour chaque activité ===

df['athlete'] = df['athlete'].apply(ast.literal_eval) # colonne pas bien formatée (ça devrait être un dictionnaire mais c'est un str)
df = df.rename(columns = {'athlete' : 'athlete_id', 'id' : 'activity_id'})
df['athlete_id'] = df['athlete_id'].apply(lambda x: x['id'] if isinstance(x, dict) else None)
df['moving_time'] = df['moving_time'] / 60 # c'est en secondes de base donc on passe en minutes
df['average_speed_km_h'] = df['average_speed'] * 3.6 # vitesse en km/h
df['average_speed_min_km'] = 60 / df['average_speed_km_h'] # vitesse en min/km
df['max_speed_km_h_activity'] = df['max_speed'] * 3.6 # vitesse en km/h
df['max_speed_min_km_activity'] = 60 / df['max_speed_km_h_activity'] # vitesse en min/km
df = df.drop(columns=["average_speed", 'max_speed']) # Puis on les enlève car on en n'a plus besoin
df['start_date'] = pd.to_datetime(df['start_date'])
# Car les colonnes dans les splits sont aussi nommées comme ça -->
df = df.rename(columns = {'distance' : 'distance_activity', 'moving_time' : 'moving_time_activity', 'average_speed' : 'average_speed_activity', 'average_speed_km_h' : 'average_speed_km_h_activity',
                          'average_speed_min_km' : 'average_speed_min_km_activity', 'average_heartrate' : 'average_heartrate_activity',
                          'total_elevation_gain' : 'elevation_gain_activity', 'average_heartrate' : 'average_heartrate_activity', 'max_heartrate' : 'max_heartrate_activity',
                          'max_speed' : 'max_speed_activity', 'average_watts' : 'average_watts_activity', 'max_watts' : 'max_watts_activity', 'weighted_average_watts' : 'weighted_average_watts_activity'})
# Car les colonnes dans les splits sont aussi nommées comme ça

# Trier les données par athlète et par date
df = df.sort_values(by=['athlete_id', 'start_date'])

# Fonction pour calculer la distance cumulée pour un sport donné
def cumulative_distance(df, sport):
    return (
        df.groupby('athlete_id', group_keys=False)
        .apply(lambda x: x.loc[x['sport_type'] == sport, 'distance_activity'].cumsum())
        .squeeze()  # Convertit le DataFrame en Series
    )

# Appliquer la fonction pour chaque sport
df['cumulative_distance_run'] = cumulative_distance(df, 'Run')
df['cumulative_distance_ride'] = cumulative_distance(df, 'Ride')
df['cumulative_distance_swim'] = cumulative_distance(df, 'Swim')

# Convertir en liste de dictionnaires
df["splits_metric"] = df["splits_metric"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explosion des splits puis des laps
df_splits_metric = df.explode("splits_metric").reset_index(drop=True)

# Extraction des valeurs de chaque split (puis laps) dans des colonnes distinctes
df_splits_metric = pd.concat([df_splits_metric.drop(columns=["splits_metric"]), df_splits_metric["splits_metric"].apply(pd.Series)], axis=1)
df = df.drop(columns=["splits_metric"]) # Puis on l'enlève car on en n'a plus besoin

df_laps = df.explode("laps").reset_index(drop=True) # Explosion des laps
df_laps = pd.concat([df_laps.drop(columns=["laps"]), df_laps["laps"].apply(pd.Series)], axis=1)
df = df.drop(columns=["laps"]) # Puis on l'enlève car on en n'a plus besoin

# === Data processing pour les splits (df_splits_metric) ===

df_splits_metric['start_date'] = pd.to_datetime(df_splits_metric['start_date'])
df_splits_metric = df_splits_metric.rename(columns = {'distance' : 'distance_split', 'moving_time' : 'moving_time_split', 'average_speed' : 'average_speed_split',
                                                      'average_speed_km_h' : 'average_speed_km_h_split', 'average_speed_min_km' : 'average_speed_min_km_split',
                                                      'average_heartrate' : 'average_heartrate_split'}) 


# === Extraction des splits et laps dans df_splits_metric ===

df_split_laps = df_splits_metric.explode("laps").reset_index(drop=True) # Explosion des laps
df_split_laps = pd.concat([df_split_laps.drop(columns=["laps"]), df_split_laps["laps"].apply(pd.Series)], axis=1)
df_splits_metric = df_splits_metric.drop(columns=["laps"]) # Puis on l'enlève car on en n'a plus besoin

df_split_split = df_splits_metric.explode("split").reset_index(drop=True) # Explosion des splits
df_split_split = pd.concat([df_split_split.drop(columns=["split"]), df_split_split["split"].apply(pd.Series)], axis=1)
df_splits_metric = df_splits_metric.drop(columns=["split"]) # Puis on l'enlève car on en n'a plus besoin

C:\Users\Mathi\AppData\Local\Temp\ipykernel_24308\442885847.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['sport_type'] == sport, 'distance_activity'].cumsum())
C:\Users\Mathi\AppData\Local\Temp\ipykernel_24308\442885847.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['sport_type'] == sport, 'distance_activity'].cumsum())
C:\Users\Mathi\AppData\L

In [62]:
print(df.columns)
print(len(df))

Index(['athlete_id', 'distance_activity', 'moving_time_activity',
       'elevation_gain_activity', 'sport_type', 'activity_id', 'start_date',
       'average_heartrate_activity', 'max_heartrate_activity',
       'average_watts_activity', 'max_watts_activity',
       'weighted_average_watts_activity', 'average_speed_km_h_activity',
       'average_speed_min_km_activity', 'max_speed_km_h_activity',
       'max_speed_min_km_activity', 'cumulative_distance_run',
       'cumulative_distance_ride', 'cumulative_distance_swim'],
      dtype='object')
945


In [63]:
df.tail()

,athlete_id,distance_activity,moving_time_activity,elevation_gain_activity,sport_type,activity_id,start_date,average_heartrate_activity,max_heartrate_activity,average_watts_activity,max_watts_activity,weighted_average_watts_activity,average_speed_km_h_activity,average_speed_min_km_activity,max_speed_km_h_activity,max_speed_min_km_activity,cumulative_distance_run,cumulative_distance_ride,cumulative_distance_swim
929,118945026,7051.6,23.050000,30.0,Ride,14097820954,2025-04-06 12:29:23+00:00,132.8,155.0,NaN,NaN,NaN,18.3564,3.268615,32.256,1.860119,NaN,898910.1,NaN
928,118945026,6750.7,29.683333,37.0,Run,14097823367,2025-04-06 12:59:32+00:00,155.1,167.0,360.8,485.0,342.0,13.6440,4.397537,18.720,3.205128,3128928.9,NaN,NaN
927,118945026,6011.2,28.350000,91.0,Run,14129225169,2025-04-09 17:02:16+00:00,147.3,171.0,357.8,576.0,347.0,12.7224,4.716091,23.904,2.510040,3134940.1,NaN,NaN
944,118945026,2240.8,12.700000,17.0,Run,14161505025,2025-04-13 05:54:16+00:00,129.0,143.0,298.9,485.0,270.0,10.5876,5.667007,15.768,3.805175,3137180.9,NaN,NaN
943,118945026,42496.3,192.616667,207.0,Run,14163073729,2025-04-13 06:30:10+00:00,168.5,186.0,356.3,558.0,363.0,13.2372,4.532681,19.152,3.132832,3179677.2,NaN,NaN


In [64]:
df_splits_metric = df_splits_metric.drop(columns = ['elapsed_time', 'elevation_difference', 0, 'average_grade_adjusted_speed', 'cumulative_distance_swim',
                                                    'cumulative_distance_ride', 'cumulative_distance_run', 'pace_zone'])

In [65]:
df_splits_metric.columns

Index(['athlete_id', 'distance_activity', 'moving_time_activity',
       'elevation_gain_activity', 'sport_type', 'activity_id', 'start_date',
       'average_heartrate_activity', 'max_heartrate_activity',
       'average_watts_activity', 'max_watts_activity',
       'weighted_average_watts_activity', 'average_speed_km_h_activity',
       'average_speed_min_km_activity', 'max_speed_km_h_activity',
       'max_speed_min_km_activity', 'distance_split', 'moving_time_split',
       'average_speed_split', 'average_heartrate_split'],
      dtype='object')

In [66]:
df_splits_metric.tail()

,athlete_id,distance_activity,moving_time_activity,elevation_gain_activity,sport_type,activity_id,start_date,average_heartrate_activity,max_heartrate_activity,average_watts_activity,max_watts_activity,weighted_average_watts_activity,average_speed_km_h_activity,average_speed_min_km_activity,max_speed_km_h_activity,max_speed_min_km_activity,distance_split,moving_time_split,average_speed_split,average_heartrate_split
4644,118945026,42496.3,192.616667,207.0,Run,14163073729,2025-04-13 06:30:10+00:00,168.5,186.0,356.3,558.0,363.0,13.2372,4.532681,19.152,3.132832,999.8,308.0,3.25,167.454545
4645,118945026,42496.3,192.616667,207.0,Run,14163073729,2025-04-13 06:30:10+00:00,168.5,186.0,356.3,558.0,363.0,13.2372,4.532681,19.152,3.132832,999.1,318.0,3.14,164.871069
4646,118945026,42496.3,192.616667,207.0,Run,14163073729,2025-04-13 06:30:10+00:00,168.5,186.0,356.3,558.0,363.0,13.2372,4.532681,19.152,3.132832,1000.1,320.0,3.13,161.412500
4647,118945026,42496.3,192.616667,207.0,Run,14163073729,2025-04-13 06:30:10+00:00,168.5,186.0,356.3,558.0,363.0,13.2372,4.532681,19.152,3.132832,999.4,302.0,3.31,159.301325
4648,118945026,42496.3,192.616667,207.0,Run,14163073729,2025-04-13 06:30:10+00:00,168.5,186.0,356.3,558.0,363.0,13.2372,4.532681,19.152,3.132832,495.1,119.0,4.16,169.907563


In [68]:
total_distance = df_splits_metric.groupby('athlete_id')['distance_split'].sum()
total_distance = pd.DataFrame(total_distance).reset_index()
total_distance = total_distance.rename(columns = {'distance_total_activity' : 'total_distance'})
total_distance

,athlete_id,distance_split
0,111468957,952117.0
1,118945026,2892526.4


In [ ]:
unique_counts = df_splits_metric.nunique().sort_values(ascending=False) # Compte le nombre de valeurs uniques dans chaque colonne
df_unique = pd.DataFrame({'Colonne': unique_counts.index, 'Valeurs Uniques': unique_counts.values})
print(df_unique)

                         Colonne  Valeurs Uniques
0              average_heartrate             3765
1                    activity_id              943
2                     start_date              939
3           average_speed_min_km              868
4                    moving_time              814
5                       distance              795
6        distance_total_activity              793
7                  average_speed              782
8                      max_speed              704
9             average_speed_km_h              657
10                 average_speed              657
11       cumulative_distance_run              656
12                  elapsed_time              635
13                   moving_time              494
14             average_heartrate              470
15          elevation_difference              449
16                          name              441
17                          laps              429
18                      calories              350


In [57]:
missing_values_count = df_splits_metric.isnull().sum() # Compte le nombre de valeurs nulles pour chaque variable
missing_values_count.sort_values(ascending=False)

0                               4603
cumulative_distance_swim        4472
cumulative_distance_ride        3636
average_temp                    3230
max_watts                       2247
weighted_average_watts          2247
average_watts                   1982
average_grade_adjusted_speed    1687
best_efforts                    1683
cumulative_distance_run         1298
average_cadence                 1095
average_heartrate                797
distance                         583
moving_time                      583
split                            583
average_speed                    583
pace_zone                        583
laps                             514
calories                         506
available_zones                  506
max_heartrate                    289
average_heartrate                289
athlete_count                      0
distance_total_activity            0
moving_time                        0
total_elevation_gain               0
sport_type                         0
a

In [8]:
df_values = df["athlete_id"].value_counts().reset_index() # Compte le nombre d'occcurences différentes pour chaque valeur d'une certaine variable
df_values.columns = ["Valeur", "Occurrences"]
df_values.head()

,Valeur,Occurrences
0,118945026,493
1,111468957,450


In [ ]:
df_run = df[df['sport_type'] == 'Run'] # On divise on 4 datasets, pour chaque type d'activité
df_bike = df[df['sport_type'] == 'Ride']
df_swim = df[df['sport_type'] == 'Swim']
df_workout = df[df['sport_type'] == 'Workout']

In [ ]:
df_run = df_run.drop(columns = ['visibility', 'private', 'sport_type'])
df_bike = df_bike.drop(columns = ['visibility', 'private', 'sport_type'])
df_swim = df_swim.drop(columns = ['visibility', 'private', 'sport_type'])
df_workout = df_workout.drop(columns = ['visibility', 'private', 'sport_type'])

## #1 Running activites analysis ###

### Univariate analysis

In [ ]:
df_run.dtypes

In [ ]:
unique_counts_run = df_run.nunique().sort_values(ascending=False)
df_unique_run = pd.DataFrame({'Colonne': unique_counts_run.index, 'Valeurs Uniques': unique_counts_run.values})
print(df_unique_run)

In [ ]:
numerical_var = df_run.select_dtypes(include=['number'])

fig = plt.figure(figsize=(18, 16))

for index, col in enumerate(numerical_var.columns, 1):
    plt.subplot(6, 4, index)
    sns.histplot(df_run[col], kde=False)
    plt.title(col)

fig.tight_layout(pad=1.0)
plt.show()

In [ ]:
categorical_var = df_run.select_dtypes(exclude=['number'])
categorical_var = categorical_var.drop(columns=['start_date']) # pas ouf pour avec ces data, juste gear_id d'intéressant
fig = plt.figure(figsize=(18, 16))

for index, col in enumerate(categorical_var.columns, 1):
    plt.subplot(6, 4, index)
    sns.countplot(df_run[col])
    plt.xticks(rotation = 90)
    plt.title(col)

fig.tight_layout(pad=1.0)
plt.show()

### Bivariate analysis

In [ ]:
plt.figure(figsize=(10,6))
correlation = numerical_var.corr()
sns.heatmap(correlation, linewidths=0.5, cmap='Blues', annot=True)